# Faster R-CNN — COCO Pretrained · Helmet Detection
## ⚠️ SADECE 1. CELL'İ DOLDUR — GERİSİNE DOKUNMA

**Model:** Faster R-CNN ResNet50-FPN — COCO pretrained → 3-class fine-tune  
**Classes:** 0=background · 1=Helmet · 2=Head (No Helmet)  
**Split:** 80/10/10 · seed=42  

---
**Yapman gereken:**
1. Aşağıdaki cell'e Kaggle bilgilerini ve Drive yolunu yaz
2. Runtime → Change runtime type → **GPU** seç
3. Runtime → **Run all**
4. Bitti — sonuçlar Drive'a kaydedilir

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║         SADECE BU CELL'İ DOLDUR — GERİSİNE DOKUNMA         ║
# ╚══════════════════════════════════════════════════════════════╝

# 1) Kaggle API bilgilerin
#    kaggle.com → Account → API → Create New Token → kaggle.json içindeki değerler
KAGGLE_USERNAME = 'KAGGLE_KULLANICI_ADIN'   # örn: 'ahmetyilmaz'
KAGGLE_KEY      = 'KAGGLE_API_KEYIN'         # örn: 'abc123def456...'

# 2) Drive'daki Roboflow PPE dataset'inin train klasörü
#    (images/ ve labels/ klasörlerini içeren klasör)
#    Drive'da sağ tık → 'Copy path' yapabilirsin
ROBOFLOW_TRAIN_DIR = '/content/drive/Othercomputers/MacBook Pro\'m/Desktop/ppe detection.yolov8/train'

# 3) Sonuçların kaydedileceği Drive klasörü
DRIVE_SAVE_DIR = '/content/drive/MyDrive/rcnn_coco_results'

In [ ]:
# ── Kurulum (dokunma) ─────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

import subprocess, os, json
subprocess.run(['pip', 'install', '-q', 'kaggle', 'pycocotools', 'torchmetrics'])

os.makedirs('/root/.kaggle', exist_ok=True)
with open('/root/.kaggle/kaggle.json', 'w') as f:
    json.dump({'username': KAGGLE_USERNAME, 'key': KAGGLE_KEY}, f)
os.chmod('/root/.kaggle/kaggle.json', 0o600)
os.makedirs(DRIVE_SAVE_DIR, exist_ok=True)
print('Kurulum OK.')

In [ ]:
# ── Veri hazırlama — 80/10/10 split (dokunma) ────────────────
import subprocess, shutil, random
import xml.etree.ElementTree as ET
from pathlib import Path

os.makedirs('/content/kaggle_raw', exist_ok=True)
subprocess.run([
    'kaggle', 'datasets', 'download',
    '-d', 'andrewmvd/hard-hat-detection',
    '--unzip', '-p', '/content/kaggle_raw', '-q'
])

CLASS_MAP = {'helmet': 0, 'head': 1}

def xml_to_yolo_lines(xml_path, W, H):
    lines = []
    for obj in ET.parse(xml_path).getroot().findall('object'):
        name = obj.find('name').text.lower().strip()
        if name not in CLASS_MAP:
            continue
        bb = obj.find('bndbox')
        xmin = float(bb.find('xmin').text)
        ymin = float(bb.find('ymin').text)
        xmax = float(bb.find('xmax').text)
        ymax = float(bb.find('ymax').text)
        if xmax <= xmin or ymax <= ymin:
            continue
        cx = round((xmin + xmax) / 2 / W, 6)
        cy = round((ymin + ymax) / 2 / H, 6)
        bw = round((xmax - xmin) / W, 6)
        bh = round((ymax - ymin) / H, 6)
        lines.append(f"{CLASS_MAP[name]} {cx} {cy} {bw} {bh}")
    return lines

all_items = []

# Kaggle XML
for xml_path in sorted(Path('/content/kaggle_raw/annotations').glob('*.xml')):
    img_path = Path('/content/kaggle_raw/images') / (xml_path.stem + '.png')
    if not img_path.exists():
        continue
    root = ET.parse(str(xml_path)).getroot()
    size = root.find('size')
    if size is None:
        continue
    W = int(size.find('width').text)
    H = int(size.find('height').text)
    if W == 0 or H == 0:
        continue
    lines = xml_to_yolo_lines(str(xml_path), W, H)
    if lines:
        all_items.append((str(img_path), lines))

# Roboflow YOLO TXT
roboflow_images = Path(ROBOFLOW_TRAIN_DIR + '/images')
roboflow_labels = Path(ROBOFLOW_TRAIN_DIR + '/labels')
for img_path in sorted(roboflow_images.glob('*.*')):
    if img_path.suffix.lower() not in ['.jpg', '.jpeg', '.png']:
        continue
    lbl_path = roboflow_labels / (img_path.stem + '.txt')
    if not lbl_path.exists():
        continue
    content = lbl_path.read_text().strip()
    if content:
        all_items.append((str(img_path), content.split('\n')))

print(f'Toplam ornek: {len(all_items)}')

# 80 / 10 / 10
random.seed(42)
random.shuffle(all_items)
n       = len(all_items)
n_train = int(n * 0.8)
n_val   = int(n * 0.1)
splits  = {
    'train': all_items[:n_train],
    'valid': all_items[n_train : n_train + n_val],
    'test' : all_items[n_train + n_val :],
}

for split, items in splits.items():
    img_dir = f'/content/merged/{split}/images'
    lbl_dir = f'/content/merged/{split}/labels'
    os.makedirs(img_dir, exist_ok=True)
    os.makedirs(lbl_dir, exist_ok=True)
    for img_path, lines in items:
        p = Path(img_path)
        shutil.copy(img_path, f'{img_dir}/{p.name}')
        with open(f'{lbl_dir}/{p.stem}.txt', 'w') as f:
            f.write('\n'.join(lines))
    print(f'  {split}: {len(items)}')

shutil.rmtree('/content/kaggle_raw')
print('Veri hazir!')

In [ ]:
# ── Dataset + DataLoader (dokunma) ───────────────────────────
import torch, cv2
import torchvision
import torchvision.transforms.functional as TF
from torch.utils.data import Dataset, DataLoader

class HelmetDataset(Dataset):
    def __init__(self, img_dir, lbl_dir, augment=False):
        self.img_dir = img_dir
        self.lbl_dir = lbl_dir
        self.augment = augment
        self.files   = sorted([
            f for f in os.listdir(img_dir)
            if f.lower().endswith(('.jpg', '.jpeg', '.png'))
        ])

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):
        fname = self.files[idx]
        img   = cv2.imread(os.path.join(self.img_dir, fname))
        img   = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        H, W  = img.shape[:2]

        lbl_path = os.path.join(self.lbl_dir, os.path.splitext(fname)[0] + '.txt')
        boxes, labels = [], []
        if os.path.exists(lbl_path):
            for line in open(lbl_path).readlines():
                parts = line.strip().split()
                if len(parts) != 5:
                    continue
                cls_id = int(parts[0]) + 1
                cx, cy, bw, bh = map(float, parts[1:])
                xmin = max(0.0, (cx - bw / 2) * W)
                ymin = max(0.0, (cy - bh / 2) * H)
                xmax = min(float(W), (cx + bw / 2) * W)
                ymax = min(float(H), (cy + bh / 2) * H)
                if xmax > xmin + 1 and ymax > ymin + 1:
                    boxes.append([xmin, ymin, xmax, ymax])
                    labels.append(cls_id)

        if self.augment and boxes and torch.rand(1).item() > 0.5:
            img   = img[:, ::-1, :].copy()
            boxes = [[W - b[2], b[1], W - b[0], b[3]] for b in boxes]

        if not boxes:
            return None

        boxes_t = torch.as_tensor(boxes, dtype=torch.float32)
        target  = {
            'boxes':    boxes_t,
            'labels':   torch.as_tensor(labels, dtype=torch.int64),
            'image_id': torch.tensor([idx]),
            'area':     (boxes_t[:, 3] - boxes_t[:, 1]) * (boxes_t[:, 2] - boxes_t[:, 0]),
            'iscrowd':  torch.zeros(len(labels), dtype=torch.int64),
        }
        return TF.to_tensor(img), target

def collate_fn(batch):
    batch = [b for b in batch if b is not None]
    return tuple(zip(*batch))

train_ds = HelmetDataset('/content/merged/train/images', '/content/merged/train/labels', augment=True)
val_ds   = HelmetDataset('/content/merged/valid/images', '/content/merged/valid/labels')
test_ds  = HelmetDataset('/content/merged/test/images',  '/content/merged/test/labels')

train_loader = DataLoader(train_ds, batch_size=4, shuffle=True,  num_workers=2, collate_fn=collate_fn)
val_loader   = DataLoader(val_ds,   batch_size=4, shuffle=False, num_workers=2, collate_fn=collate_fn)
test_loader  = DataLoader(test_ds,  batch_size=4, shuffle=False, num_workers=2, collate_fn=collate_fn)

device = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')
print(f'Device : {device}')
print(f'Train  : {len(train_ds)}')
print(f'Val    : {len(val_ds)}')
print(f'Test   : {len(test_ds)}')

In [ ]:
# ── Model (dokunma) ──────────────────────────────────────────
from torchvision.models.detection import fasterrcnn_resnet50_fpn, FasterRCNN_ResNet50_FPN_Weights
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor

model = fasterrcnn_resnet50_fpn(weights=FasterRCNN_ResNet50_FPN_Weights.COCO_V1)
in_features = model.roi_heads.box_predictor.cls_score.in_features
model.roi_heads.box_predictor = FastRCNNPredictor(in_features, 3)
model.to(device)
print('Model: Faster R-CNN ResNet50-FPN — COCO pretrained — 3 class head')
print(f'Toplam parametre: {sum(p.numel() for p in model.parameters()):,}')

In [ ]:
# ── Eğitim — warmup + StepLR + grad clip + val mAP (dokunma) ─
import time
from torch.optim.lr_scheduler import LinearLR, StepLR
from torchmetrics.detection.mean_ap import MeanAveragePrecision

EPOCHS       = 10
EVAL_EVERY   = 2
LR           = 0.005
WARMUP_STEPS = 300
GRAD_CLIP    = 1.0
CLASS_NAMES  = ['helmet', 'head']

optimizer        = torch.optim.SGD([p for p in model.parameters() if p.requires_grad],
                                    lr=LR, momentum=0.9, weight_decay=0.0005)
warmup_scheduler = LinearLR(optimizer, start_factor=0.01, end_factor=1.0, total_iters=WARMUP_STEPS)
main_scheduler   = StepLR(optimizer, step_size=5, gamma=0.5)

def compute_map(loader):
    model.eval()
    metric = MeanAveragePrecision(iou_type='bbox', class_metrics=True)
    with torch.no_grad():
        for images, targets in loader:
            images  = [img.to(device) for img in images]
            outputs = model(images)
            preds = [{'boxes': o['boxes'].cpu(), 'scores': o['scores'].cpu(),
                      'labels': o['labels'].cpu()} for o in outputs]
            gts   = [{'boxes': t['boxes'].cpu(), 'labels': t['labels'].cpu()} for t in targets]
            metric.update(preds, gts)
    r = metric.compute()
    per_class = {CLASS_NAMES[i]: float(ap) for i, ap in enumerate(r.get('map_per_class', []))}
    return float(r['map']), float(r['map_50']), float(r['map_75']), per_class

best_map50  = 0.0
best_loss   = float('inf')
global_step = 0
history     = []
nan_count   = 0

print(f'Egitim basliyor — {EPOCHS} epoch  device={device}')
print('=' * 70)

for epoch in range(1, EPOCHS + 1):
    model.train()
    epoch_loss = 0.0
    valid_steps = 0
    t0 = time.time()

    for step, (images, targets) in enumerate(train_loader, 1):
        images  = [img.to(device) for img in images]
        targets = [{k: v.to(device) for k, v in t.items()} for t in targets]
        losses  = sum(model(images, targets).values())

        if not torch.isfinite(losses):
            nan_count += 1
            optimizer.zero_grad()
            continue

        optimizer.zero_grad()
        losses.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
        optimizer.step()

        global_step += 1
        if global_step <= WARMUP_STEPS:
            warmup_scheduler.step()

        epoch_loss  += losses.item()
        valid_steps += 1

        if step % 100 == 0:
            lr_now = optimizer.param_groups[0]['lr']
            print(f'  Ep [{epoch}/{EPOCHS}] Step [{step}/{len(train_loader)}] '
                  f'Loss: {losses.item():.4f}  LR: {lr_now:.6f}')

    avg_loss = epoch_loss / max(valid_steps, 1)
    lr_now   = optimizer.param_groups[0]['lr']
    if global_step > WARMUP_STEPS:
        main_scheduler.step()

    map_all = map50 = map75 = None
    per_class = {}
    if epoch % EVAL_EVERY == 0:
        print(f'  Val mAP hesaplaniyor (epoch {epoch})...')
        t_eval = time.time()
        map_all, map50, map75, per_class = compute_map(val_loader)
        print(f'  Val mAP@0.50={map50:.4f}  mAP@0.50:0.95={map_all:.4f}  ({time.time()-t_eval:.0f}s)')
        print(f'    ' + '  '.join(f'{k}={v:.4f}' for k, v in per_class.items()))

    is_best = (map50 is not None and map50 > best_map50) or \
              (map50 is None and avg_loss < best_loss)
    if map50 is not None and map50 > best_map50:
        best_map50 = map50
    if avg_loss < best_loss:
        best_loss = avg_loss

    ckpt = {'epoch': epoch, 'model_state_dict': model.state_dict(),
            'loss': avg_loss, 'map50': map50}
    torch.save(ckpt, '/content/checkpoint_latest.pth')
    if is_best:
        torch.save(ckpt, '/content/best_model_coco.pth')
        tag = f'mAP@0.5={best_map50:.4f}' if map50 else f'loss={avg_loss:.4f}'
        print(f'  -> Best model kaydedildi ({tag})')

    entry = {'epoch': epoch, 'loss': avg_loss, 'lr': lr_now, 'nan_skipped': nan_count}
    if map50 is not None:
        entry.update({'map': map_all, 'map_50': map50, 'map_75': map75, 'per_class': per_class})
    history.append(entry)
    with open('/content/train_history_coco.json', 'w') as f:
        json.dump(history, f, indent=2)

    map_str = f'  mAP@0.5={map50:.4f}' if map50 else ''
    print(f'Epoch [{epoch}/{EPOCHS}]  Loss={avg_loss:.4f}{map_str}  ({time.time()-t0:.0f}s)')
    print('=' * 70)

print(f'Egitim tamamlandi. En iyi val mAP@0.50: {best_map50:.4f}')

In [ ]:
# ── Test seti + FPS (dokunma) ─────────────────────────────────
ckpt = torch.load('/content/best_model_coco.pth', map_location=device)
model.load_state_dict(ckpt['model_state_dict'])
print(f"En iyi model yuklendi (epoch={ckpt['epoch']}, val_map50={ckpt['map50']})")

print('\nTest seti degerlendiriliyor...')
test_map, test_map50, test_map75, test_per_class = compute_map(test_loader)

model.eval()
total_time = total_images = 0
with torch.no_grad():
    for images, _ in test_loader:
        images = [img.to(device) for img in images]
        t0 = time.time()
        _ = model(images)
        total_time   += time.time() - t0
        total_images += len(images)
fps = total_images / total_time

print('\n' + '=' * 55)
print('  Faster R-CNN (COCO Pretrained) - TEST SONUCLARI')
print('=' * 55)
print(f'  mAP@0.50       : {test_map50:.4f}  ({test_map50*100:.2f}%)')
print(f'  mAP@0.50:0.95  : {test_map:.4f}  ({test_map*100:.2f}%)')
print(f'  mAP@0.75       : {test_map75:.4f}')
print('-' * 55)
for cls, ap in test_per_class.items():
    print(f'  {cls:<12}: {ap:.4f}')
print('-' * 55)
print(f'  FPS (GPU)      : {fps:.1f}')
print('=' * 55)

results = {
    'model': 'Faster R-CNN (COCO pretrained)',
    'best_epoch': ckpt['epoch'],
    'val_map50_best': ckpt['map50'],
    'test': {'map': test_map, 'map_50': test_map50, 'map_75': test_map75,
             'per_class_ap': test_per_class, 'fps': fps}
}
with open('/content/results_coco.json', 'w') as f:
    json.dump(results, f, indent=2)

In [ ]:
# ── Drive'a kaydet (dokunma) ──────────────────────────────────
import shutil
for src, fname in [
    ('/content/best_model_coco.pth',     'best_model_coco.pth'),
    ('/content/train_history_coco.json', 'train_history_coco.json'),
    ('/content/results_coco.json',       'results_coco.json'),
]:
    dst = os.path.join(DRIVE_SAVE_DIR, fname)
    shutil.copy(src, dst)
    print(f'Kaydedildi: {dst}')

print('\n--- SONUC OZETI ---')
print(f"Val  mAP@0.50 (en iyi epoch) : {ckpt['map50']:.4f}  ({ckpt['map50']*100:.2f}%)")
print(f'Test mAP@0.50                : {test_map50:.4f}  ({test_map50*100:.2f}%)')
print(f'FPS                          : {fps:.1f}')